# Data analysis

### Plot results from hierarchical_model_train_rfecv.py execution in hpc

### Cell 1 — Hierarchical run: confusion matrix heatmap (single subset)

Loads results from a single HPC run directory (`results_root`) and iterates over its `rfecv_*` subdirectories.

For each subset it:
1. **Detects the prediction mode** — `binary`, `multiclass`, or `hierarchical` (both confusion matrix files present).
2. **Loads the predictions CSV** and, for hierarchical mode, filters out `NEGATIVE` samples then computes the micro-averaged F1 score on the remaining phenotype predictions.
3. **Loads the confusion matrix CSV** (normalised values) and strips verbose label prefixes for multiclass runs.
4. **Reads `summary.json`** for ROC-AUC and other saved metrics.
5. **Plots a heatmap** (blue for binary, green for multiclass/hierarchical) with the AUC in the title.

The `break` at the end limits the output to the first subset — remove it to iterate over all subsets.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import f1_score

results_root = Path(
    "/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/binary_optuna_cef_new_20260202172017/"
)

subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None


for subset_dir in subset_dirs:
    binary_cm = subset_dir / "binary_confusion_matrix.csv"
    multiclass_cm = subset_dir / "multiclass_confusion_matrix.csv"

    # Detect mode — check both files before assigning to correctly identify hierarchical
    if binary_cm.exists() and multiclass_cm.exists():
        mode = "hierarchical"
        cm_path = multiclass_cm  # plot the multiclass head for hierarchical runs
    elif binary_cm.exists():
        mode = "binary"
        cm_path = binary_cm
    elif multiclass_cm.exists():
        mode = "multiclass"
        cm_path = multiclass_cm
    else:
        print(f"No confusion matrix found in {subset_dir}")
        continue

    predictions_dirs = [f for f in subset_dir.iterdir() if "predictions.csv" in f.name]
    if len(predictions_dirs) == 1:
        predictions_df = pd.read_csv(predictions_dirs[0])
    elif len(predictions_dirs) > 1:
        raise ValueError("MULTIPLE PREDICTIONS CSV FOUND: ", str(predictions_dirs))
    else:
        predictions_df = None
        print("NO PREDICTION DIR FOUND")

    f1_score_m = None
    if predictions_df is not None and mode == "hierarchical":
        pos_mask = (
            (predictions_df["true_fenotipo"] != "NEGATIVE")
            & (predictions_df["hierarchical_pred"] != "NEGATIVE")
        )
        pos_df = predictions_df[pos_mask]
        if not pos_df.empty:
            sw = pos_df["sample_weight"] if "sample_weight" in pos_df.columns else None
            f1_score_m = f1_score(
                pos_df["true_fenotipo"],
                pos_df["hierarchical_pred"],
                average="micro",
                sample_weight=sw,
            )
            print(f"Hierarchical micro F1 (positive samples only): {f1_score_m:.3f}")

    df = load_confusion_df(cm_path)
    if df is None:
        print(f"Empty confusion matrix in {subset_dir}")
        continue

    # Strip verbose label prefixes (relevant for multiclass / hierarchical)
    if mode in ("multiclass", "hierarchical"):
        df = df.rename(
            columns=lambda x: x.split("resistente a ")[-1],
            index=lambda x: x.split("resistente a ")[-1],
        )

    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    if f1_score_m is not None:
        summary["f1_score_micro"] = f1_score_m

    if mode == "binary":
        auc_val = summary.get("binary_roc_auc")
        suffix = f"\nBinary macro ROC-AUC: {auc_val:.3f}" if auc_val is not None else ""
        cmap = "Blues"
        title = f"{subset_dir.name} – Binary classification{suffix}"
    else:
        auc_val = summary.get("multiclass_macro_auc")
        suffix = f"\nMulticlass macro ROC-AUC: {auc_val:.3f}" if auc_val is not None else ""
        if f1_score_m is not None:
            suffix += f" | micro F1: {f1_score_m:.3f}"
        label = "Hierarchical – Multiclass head" if mode == "hierarchical" else "Multiclass classification"
        cmap = "Greens"
        title = f"{subset_dir.name} – {label}{suffix}"

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(df, annot=True, fmt=".2f", cmap=cmap, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    break  # remove to iterate over all subsets
else:
    print("subset_dirs is empty")

### Cell 2 — Batch scan: binary ROC-AUC summary across cefalosporinas runs

Scans **all** run directories under the shared outputs root that match `binary_optuna_cefalosporinas` or `other` in their name.

For each matching run it iterates over `rfecv_*` subdirectories, loads the confusion matrix and `summary.json`, then prints the run name together with the best binary model identifier and its ROC-AUC score. Useful for a quick comparison of multiple experiment runs without plotting.

In [ ]:
aucs, f1s = [], []
for root_dir in Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/").iterdir():

    if not "binary_optuna_cefalosporinas" in root_dir.name and not "other" in root_dir.name:
        continue
    subset_dirs = sorted(
        d for d in root_dir.iterdir()
        if d.is_dir() and d.name.startswith("rfecv_")
    )
    
    def load_confusion_df(path: Path) -> pd.DataFrame | None:
        return pd.read_csv(path, index_col=0) if path.exists() else None


    for subset_dir in subset_dirs:
        # --- detectar modo por nombre de archivo ---
        binary_cm = subset_dir / "binary_confusion_matrix.csv"
        multiclass_cm = subset_dir / "multiclass_confusion_matrix.csv"

        if binary_cm.exists():
            mode = "binary"
            cm_path = binary_cm
        elif multiclass_cm.exists():
            mode = "multiclass"
            cm_path = multiclass_cm
        else:
            print(f"No confusion matrix found in {subset_dir}")
            continue

        df = load_confusion_df(cm_path)
        if df is None:
            print(f"Empty confusion matrix in {subset_dir}")
            continue

        # limpieza de etiquetas (solo relevante para multiclase)
        if mode == "multiclass":
            df = df.rename(
                columns=lambda x: x.split("resistente a ")[-1],
                index=lambda x: x.split("resistente a ")[-1],
            )

        # --- cargar summary ---
        summary_path = subset_dir / "summary.json"
        summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    print(root_dir.name)
    print(summary.get("binary_model"), summary.get("binary_roc_auc"))


### Cell 3 — Binary-only run: confusion matrix heatmap (single subset)

Targets a specific binary-only run directory, restricting to `rfecv_bin*` subdirectories.

For each subset it:
1. Loads `binary_predictions.csv` and derives the class list from the unique labels present.
2. Reads the pre-computed `binary_confusion_matrix.csv` if available; otherwise recomputes it from the raw predictions using `sklearn.metrics.confusion_matrix`.
3. Reads `summary.json` for macro F1 and ROC-AUC.
4. Plots an integer-count heatmap (blue) with both metrics in the title.

The `break` limits output to the first subset.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Point this to the run output root (e.g., outputs/binary_optuna_YYYYMMDDHHMMSS)
results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/binary_optuna_cefalosporinas_dropother_20251217120107/")
subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_bin")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None

print(subset_dirs)
for subset_dir in subset_dirs:
    preds_path = subset_dir / "binary_predictions.csv"
    if not preds_path.exists():
        print(f"Missing predictions: {preds_path}")
        continue

    preds = pd.read_csv(preds_path)
    classes = sorted(set(preds["true_binary_label"]) | set(preds["binary_pred"]))

    binary_df = load_confusion_df(subset_dir / "binary_confusion_matrix.csv")
    if binary_df is None:
        y_true = preds["true_binary_label"]
        y_pred = preds["binary_pred"]
        cm = confusion_matrix(y_true, y_pred, labels=classes)
        binary_df = pd.DataFrame(
            cm,
            index=[f"true_{lbl}" for lbl in classes],
            columns=[f"pred_{lbl}" for lbl in classes],
        )
    for col in binary_df:
        binary_df[col] = binary_df[col].astype(int)

    # Load summary for metrics
    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    auc_str = f"{summary.get('binary_roc_auc'):.3f}" if summary.get("binary_roc_auc") is not None else "N/A"
    f1_str = f"{summary.get('macro_f1'):.3f}" if summary.get("macro_f1") is not None else "N/A"

    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    sns.heatmap(binary_df, annot=True, fmt=".0f", cmap="Blues", ax=ax)
    ax.set_title(f"{subset_dir.name}\nMacro F1: {f1_str} | ROC-AUC: {auc_str}")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    break  # remove this break to loop over all subsets
else:
    print("subset_dirs is empty")


### Multiclass-only run: confusion matrix heatmap

For runs produced by `multiclass_rfecv_model_train.py` — no binary gate, direct multiclass prediction.

Reads `predictions.csv` (columns: `true_label`, `pred_label`, `confidence`) and `multiclass_confusion_matrix.csv`. Falls back to recomputing the confusion matrix from raw predictions if the CSV is missing. Metrics (`macro_f1`, `multiclass_macro_auc`) are read from `summary.json`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score

# Point to a multiclass_rfecv_model_train.py output directory
results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/outputs/multiclass_optuna_norfecv_catb_20260204123205/")
subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None

for subset_dir in subset_dirs:
    preds_path = subset_dir / "predictions.csv"
    if not preds_path.exists():
        print(f"Missing predictions: {preds_path}")
        continue

    preds = pd.read_csv(preds_path)
    classes = sorted(set(preds["true_label"]) | set(preds["pred_label"]))

    cm_df = load_confusion_df(subset_dir / "multiclass_confusion_matrix.csv")
    if cm_df is None:
        cm = confusion_matrix(preds["true_label"], preds["pred_label"], labels=classes)
        cm_df = pd.DataFrame(
            cm,
            index=[f"true_{lbl}" for lbl in classes],
            columns=[f"pred_{lbl}" for lbl in classes],
        )

    # Strip verbose label prefixes if present
    cm_df = cm_df.rename(
        columns=lambda x: x.split("resistente a ")[-1],
        index=lambda x: x.split("resistente a ")[-1],
    )

    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    auc_str = f"{summary['multiclass_macro_auc']:.3f}" if summary.get("multiclass_macro_auc") is not None else "N/A"
    f1_str = f"{summary['macro_f1']:.3f}" if summary.get("macro_f1") is not None else "N/A"

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_df, annot=True, fmt=".2f", cmap="Greens", ax=ax)
    ax.set_title(f"{subset_dir.name} – Multiclass only\nMacro F1: {f1_str} | Macro ROC-AUC: {auc_str}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()

    break  # remove to iterate over all subsets
else:
    print("subset_dirs is empty")

### Multi-label only run: 3-panel diagnostic plot

For runs produced by `model_train_bmr_multilabel_only.py` — no binary gate, direct multi-label prediction of `fenotipo_resistencia`.

Reads `multilabel_predictions.csv` (columns: `true_labels`, `pred_labels` — comma-separated) and `multilabel_metrics.json`.

**Panels:**

| Panel | Content |
|---|---|
| Left | Per-label recall — how often each true label appears in the prediction set |
| Centre | Top-8 label frequency: true count vs predicted count |
| Right | Histogram of label count per sample, annotated with *coverage* (≥1 prediction) and *hit-any* (≥1 correct) |

Saved metrics (`micro_f1`, `macro_f1`, `exact_match_ratio`, `coverage`, `inclusion_any_true`, `avg_labels_predicted`) are printed below the figure.

In [ ]:
import ast
import json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Point to a model_train_bmr_multilabel_only.py output directory
results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/multilabel_optuna_bmr_lgbm_20260206131723/")
subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_")
)

sns.set_theme(style="whitegrid")

def parse_labels(val, delim=","):
    """Parse a delimited label string into a list, handling NaN and empty strings."""
    if pd.isna(val) or str(val).strip() == "":
        return []
    return [x.strip() for x in str(val).split(delim) if x.strip()]

for subset_dir in subset_dirs:
    preds_path = subset_dir / "multilabel_predictions.csv"
    if not preds_path.exists():
        print(f"Missing predictions: {preds_path}")
        continue

    preds = pd.read_csv(preds_path)
    true_lists = preds["true_labels"].apply(parse_labels).tolist()
    pred_lists = preds["pred_labels"].apply(parse_labels).tolist()

    # Per-label recall
    true_counts = Counter(lbl for lst in true_lists for lbl in lst)
    correct_counts = Counter()
    for tl, pl in zip(true_lists, pred_lists):
        ps = set(pl)
        for lbl in tl:
            if lbl in ps:
                correct_counts[lbl] += 1
    recall_df = pd.DataFrame([
        {"label": lbl, "recall": correct_counts.get(lbl, 0) / cnt, "support": cnt}
        for lbl, cnt in true_counts.items()
    ]).sort_values("recall", ascending=False)

    # Top labels frequency comparison
    true_flat = Counter(lbl for lst in true_lists for lbl in lst)
    pred_flat = Counter(lbl for lst in pred_lists for lbl in lst)
    top_labels = list({lbl for lbl, _ in true_flat.most_common(8)} | {lbl for lbl, _ in pred_flat.most_common(8)})
    top_df = pd.DataFrame({
        "true": {lbl: true_flat.get(lbl, 0) for lbl in top_labels},
        "pred": {lbl: pred_flat.get(lbl, 0) for lbl in top_labels},
    }).reindex(index=sorted(top_labels))

    # Labels per sample + coverage / hit-any
    true_counts_per = [len(x) for x in true_lists]
    pred_counts_per = [len(x) for x in pred_lists]
    coverage = sum(c > 0 for c in pred_counts_per) / len(pred_counts_per) if pred_counts_per else 0.0
    hit_any = sum(len(set(t) & set(p)) > 0 for t, p in zip(true_lists, pred_lists)) / len(true_lists) if true_lists else 0.0

    # Load saved metrics
    ml_metrics = {}
    ml_path = subset_dir / "multilabel_metrics.json"
    if ml_path.exists():
        ml_metrics = json.loads(ml_path.read_text())

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    sns.barplot(
        data=recall_df, x="recall", y="label",
        order=recall_df["label"].tolist(), palette="Greens_r", ax=axes[0],
    )
    axes[0].set_title("Per-label recall")
    axes[0].set_xlim(0, 1)

    top_df.plot.bar(ax=axes[1], rot=45, title="Top labels: true vs predicted")
    axes[1].set_ylabel("count")

    max_count = max(true_counts_per + pred_counts_per + [1])
    axes[2].hist(true_counts_per, bins=range(0, max_count + 2), alpha=0.6, label="true")
    axes[2].hist(pred_counts_per, bins=range(0, max_count + 2), alpha=0.6, label="pred")
    axes[2].set_title(f"Labels per sample\ncoverage: {coverage:.2f} | hit-any: {hit_any:.2f}")
    axes[2].set_xlabel("# labels")
    axes[2].set_ylabel("count")
    axes[2].legend()

    if ml_metrics:
        metrics_text = "\n".join(
            [f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {str(v)[0:10] + "..." if len(str(v)) >= 10 else str(v)} "
             for k, v in ml_metrics.items()]
        )
    
        fig.text(
            1.02, 0.5, metrics_text,  # x, y (right side of figure)
            fontsize=9,
            va="center",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
        )
        print(f"\nSaved metrics for {subset_dir.name}:")
        for k, v in ml_metrics.items():
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    plt.suptitle(f"{subset_dir.name} – Multi-label only", y=1.02)
    plt.tight_layout()
    plt.show()

    break  # remove to iterate over all subsets
else:
    print("subset_dirs is empty")

### Cell 4 — Hierarchical run: binary + multiclass/multilabel side-by-side plots

Handles runs that produce both a binary gate and a downstream phenotype classifier (multiclass or multi-label).

**Setup helpers:**
- `split_labels` — parses pipe-delimited or list-formatted label strings from the predictions CSV.
- `ranked_labels_from_proba` — extracts the multiclass probability dict from the `hierarchical_proba` column and returns labels ranked by descending probability.
- `precision_recall_at_k` — computes mean precision and recall when only the top-k predicted labels are considered.

**Per-subset logic:**
1. Loads `hierarchical_predictions.csv` and determines the negative label.
2. Builds the **binary confusion matrix** (from CSV or recomputed).
3. Detects whether the run is **multi-label** (checks for `multilabel_metrics.json` or pipe-separated predictions).
4. **Multiclass path**: loads or recomputes the multiclass confusion matrix and plots it as a green heatmap.
5. **Multi-label path**: plots a bar chart of top-10 true vs predicted label frequencies, then annotates the figure with micro/macro-F1, exact match, coverage, and precision/recall@1 and @3 (computed from ranked probabilities where available, otherwise from saved metrics).
6. Renders both panels side-by-side.

The `break` limits output to the first subset.

In [ ]:
import ast
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
#hierarchical_optuna_bmr_multilabel_g2_20251216101932
results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/hierarchical_optuna_bmr_multilabel_g2_20251216101932/")
subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None

def split_labels(series: pd.Series, delim: str = "|") -> list[list[str]]:
    def parse_entry(val: object) -> list[str]:
        if isinstance(val, list):
            return [str(x) for x in val if str(x)]
        as_str = "" if pd.isna(val) else str(val)
        stripped = as_str.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(as_str)
                if isinstance(parsed, (list, tuple)):
                    return [str(x) for x in parsed if str(x)]
            except Exception:
                pass
        return [x for x in as_str.split(delim) if x]
    return series.fillna("").apply(parse_entry).tolist()

def ranked_labels_from_proba(series: pd.Series) -> list[list[str]]:
    ranked: list[list[str]] = []
    for val in series.fillna(""):
        labels: list[str] = []
        try:
            parsed = ast.literal_eval(val) if isinstance(val, str) else val
            multiclass = parsed.get("multiclass") if isinstance(parsed, dict) else {}
            if multiclass:
                labels = [lbl for lbl, _ in sorted(multiclass.items(), key=lambda x: x[1], reverse=True)]
        except Exception:
            labels = []
        ranked.append(labels)
    return ranked

def precision_recall_at_k(true_lists: list[list[str]], ranked_preds: list[list[str]], k: int) -> tuple[float, float]:
    precisions: list[float] = []
    recalls: list[float] = []
    for true_labels, pred_labels in zip(true_lists, ranked_preds):
        topk = pred_labels[:k]
        hit = len(set(true_labels) & set(topk))
        precisions.append(hit / len(topk) if topk else 0.0)
        recalls.append(hit / len(true_labels) if true_labels else 0.0)
    return (
        sum(precisions) / len(precisions) if precisions else float("nan"),
        sum(recalls) / len(recalls) if recalls else float("nan"),
    )

print(subset_dirs)
for subset_dir in subset_dirs:
    preds_path = subset_dir / "hierarchical_predictions.csv"
    if not preds_path.exists():
        print(f"Could not find {preds_path}")
        continue

    preds = pd.read_csv(preds_path)
    negative_label = next(iter(set(preds["binary_pred"]) - {"POSITIVE"}), "NEGATIVE")

    # Binary confusion
    binary_df = load_confusion_df(subset_dir / "binary_confusion_matrix.csv")
    if binary_df is None:
        y_true_bin = preds["true_binary_label"].ne(negative_label).astype(int)
        y_pred_bin = preds["binary_pred"].eq("POSITIVE").astype(int)
        cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
        binary_df = pd.DataFrame(
            cm,
            index=[f"true_{negative_label}", "true_POSITIVE"],
            columns=[f"pred_{negative_label}", "pred_POSITIVE"],
        )
    for col in binary_df:
        binary_df[col] = binary_df[col].astype(int)

    # Detect multi-label run
    multilabel_metrics_path = subset_dir / "multilabel_metrics.json"
    is_multilabel = multilabel_metrics_path.exists() or (preds["hierarchical_pred"].astype(str).str.contains(r"\|").any())

    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    binary_suffix = (
        f"\nBinary macro ROC-AUC: {summary.get('binary_roc_auc'):.3f}"
        if summary.get("binary_roc_auc") is not None
        else ""
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(binary_df, annot=True, fmt=".2f", cmap="Blues", ax=axes[0])
    axes[0].set_title(f"{subset_dir.name} – Binary{binary_suffix}")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("")

    if not is_multilabel:
        multiclass_df = load_confusion_df(subset_dir / "multiclass_confusion_matrix.csv")
        if multiclass_df is None:
            pos_mask = preds["true_binary_label"].ne(negative_label)
            multiclass_df = None
            if pos_mask.any():
                y_true_pos = preds.loc[pos_mask, "true_fenotipo"]
                y_pred_pos = preds.loc[pos_mask, "hierarchical_pred"]
                labels = sorted(set(y_true_pos) | set(y_pred_pos))
                cm = confusion_matrix(y_true_pos, y_pred_pos, labels=labels)
                multiclass_df = pd.DataFrame(
                    cm,
                    index=[f"true_{lbl}" for lbl in labels],
                    columns=[f"pred_{lbl}" for lbl in labels],
                )
        if multiclass_df is not None:
            sns.heatmap(multiclass_df, annot=True, fmt=".2f", cmap="Greens", ax=axes[1])
            axes[1].set_title(
                f"{subset_dir.name} – Multiclass "
                f"{'' if summary.get('multiclass_macro_auc') is None else f'(AUC {summary['multiclass_macro_auc']:.3f})'}"
            )
        else:
            axes[1].axis("off")
            axes[1].set_title(f"{subset_dir.name} – Multiclass")
    else:
        # Multi-label: show top label frequencies and metrics text
        pos_mask = preds["true_binary_label"].ne(negative_label)
        true_lists = split_labels(preds.loc[pos_mask, "true_fenotipo"])
        pred_lists = split_labels(preds.loc[pos_mask, "hierarchical_pred"])
        ranked_pred_lists = ranked_labels_from_proba(preds.loc[pos_mask, "hierarchical_proba"])
        true_flat = pd.Series([lbl for lst in true_lists for lbl in lst])
        pred_flat = pd.Series([lbl for lst in pred_lists for lbl in lst])
        top_true = true_flat.value_counts().head(10)
        top_pred = pred_flat.value_counts().head(10)
        freq_df = pd.DataFrame({"true": top_true, "pred": top_pred}).fillna(0).astype(int)
        freq_df.plot.bar(ax=axes[1], rot=90, title=f"{subset_dir.name} – Multi-label top-10 labels")
        axes[1].set_ylabel("count")

        ml_metrics = json.loads(multilabel_metrics_path.read_text()) if multilabel_metrics_path.exists() else {}

        precision_at_1_calc, recall_at_1_calc = precision_recall_at_k(true_lists, ranked_pred_lists, 1)
        precision_at_3_calc, recall_at_3_calc = precision_recall_at_k(true_lists, ranked_pred_lists, 3)

        precision_at_1_val = precision_at_1_calc if not pd.isna(precision_at_1_calc) else ml_metrics.get("precision_at_1", float("nan"))
        recall_at_1_val = recall_at_1_calc if not pd.isna(recall_at_1_calc) else ml_metrics.get("recall_at_1", float("nan"))
        precision_at_3_val = precision_at_3_calc if not pd.isna(precision_at_3_calc) else ml_metrics.get("precision_at_3", float("nan"))
        recall_at_3_val = recall_at_3_calc if not pd.isna(recall_at_3_calc) else ml_metrics.get("recall_at_3", float("nan"))

        text = "\n".join(
            [
                f"micro-F1: {ml_metrics.get('micro_f1', float('nan')):.3f}",
                f"macro-F1: {ml_metrics.get('macro_f1', float('nan')):.3f}",
                f"exact match: {ml_metrics.get('exact_match_ratio', float('nan')):.3f}",
                f"coverage: {ml_metrics.get('coverage', float('nan')):.3f}",
                f"inclusion_any_true: {ml_metrics.get('inclusion_any_true', float('nan')):.3f}",
                f"avg labels pred: {ml_metrics.get('avg_labels_predicted', float('nan')):.2f}",
                f"precision_at_top1: {precision_at_1_val:.2f}",
                f"recall_at_top1: {recall_at_1_val:.2f}",
                f"precision_at_top3: {precision_at_3_val:.2f}",
                f"recall_at_top3: {recall_at_3_val:.2f}",
            ]
        )
        axes[1].text(1.05, 0.5, text, transform=axes[1].transAxes, va="center")

    plt.tight_layout()
    plt.show()
    break
else:
    print("subset_dirs is empty")


### Cell 5 — Inspect `summary` dict from the previous cell

Quick display of the `summary` dictionary loaded in Cell 4. Contains persisted metrics such as `binary_roc_auc`, `multiclass_macro_auc`, `macro_f1`, and any other keys written by the training script.

In [ ]:
summary

### Cell 6 — Multi-label hierarchical run: 4-panel diagnostic dashboard

Produces a detailed 2×2 figure for a single multi-label hierarchical run.

**Panel layout:**

| Panel | Content |
|---|---|
| Top-left | Binary gate confusion matrix (integer counts, blue heatmap) |
| Top-right | Per-label recall bar chart — how well each phenotype label is recovered |
| Bottom-left | Top-7 labels: true count vs predicted count (bar chart) |
| Bottom-right | Histogram of label counts per patient for true vs predicted, annotated with *coverage* (fraction of positive patients receiving at least one prediction) and *hit-any* (fraction where at least one prediction is correct) |

**Data preparation:**
- Positive samples are isolated using `binary_pred`.
- True and predicted labels are parsed from pipe-delimited strings.
- Per-label recall is computed by counting how often each true label appears in the corresponding prediction set.
- If `multilabel_metrics.json` exists, its saved metrics (micro/macro-F1, exact match, coverage, inclusion) are printed after the figure.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid")

results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/hierarchical_optuna_bmr_multilabel_g2_20251226094705/")
subset_dirs = sorted(d for d in results_root.iterdir() if d.is_dir() and d.name.startswith("rfecv_"))
if not subset_dirs:
    raise SystemExit("No subset dirs found")
subset_dir = subset_dirs[0]  # pick the first; change if needed

preds = pd.read_csv(subset_dir / "hierarchical_predictions.csv")
negative_label = next(iter(set(preds["binary_pred"]) - {"POSITIVE"}), "NEGATIVE")

# --- Binary confusion
binary_df = pd.read_csv(subset_dir / "binary_confusion_matrix.csv", index_col=0) if (subset_dir / "binary_confusion_matrix.csv").exists() else None
if binary_df is None:
    y_true_bin = preds["true_binary_label"].ne(negative_label).astype(int)
    y_pred_bin = preds["binary_pred"].eq("POSITIVE").astype(int)
    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    binary_df = pd.DataFrame(cm,
        index=[f"true_{negative_label}", "true_POSITIVE"],
        columns=[f"pred_{negative_label}", "pred_POSITIVE"],
    )
binary_df = binary_df.astype(int)

# --- Multi-label prep (positive samples only)
pos_mask = preds["true_binary_label"].ne(negative_label)
true_lists = preds.loc[pos_mask, "true_fenotipo"].fillna("").apply(lambda s: [x for x in str(s).split("|") if x]).tolist()
pred_lists = preds.loc[pos_mask, "hierarchical_pred"].fillna("").apply(lambda s: [x for x in str(s).split("|") if x]).tolist()

# Per-label recall
true_counts = Counter(lbl for lst in true_lists for lbl in lst)

# Top labels (true vs pred)
true_flat = Counter(lbl for lst in true_lists for lbl in lst)
pred_flat = Counter(lbl for lst in pred_lists for lbl in lst)
top_true = true_flat.most_common(7)
top_pred = pred_flat.most_common(7)
top_labels = list({lbl for lbl, _ in top_true} | {lbl for lbl, _ in top_pred})
top_df = pd.DataFrame({
    "true": {lbl: true_flat.get(lbl, 0) for lbl in top_labels},
    "pred": {lbl: pred_flat.get(lbl, 0) for lbl in top_labels},
}).reindex(index=sorted(top_labels))

# Label count per patient + coverage/hit-any
true_counts_per = [len(x) for x in true_lists]
pred_counts_per = [len(x) for x in pred_lists]
coverage = sum(c > 0 for c in pred_counts_per) / len(pred_counts_per) if pred_counts_per else 0.0
hit_any = sum(len(set(t) & set(p)) > 0 for t, p in zip(true_lists, pred_lists)) / len(true_lists) if true_lists else 0.0

# Optional saved metrics
ml_metrics = {}
ml_path = subset_dir / "multilabel_metrics.json"
if ml_path.exists():
    ml_metrics = json.loads(ml_path.read_text())

# --- Plotting (4 panels)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.heatmap(binary_df, annot=True, fmt="d", cmap="Blues", ax=axes[0, 0])
axes[0, 0].set_title(f"{subset_dir.name} – Binary gate")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("")

axes[0, 1].axis("off")

top_df.plot.bar(ax=axes[1, 0], rot=45, title="Top labels: true vs predicted")
axes[1, 0].set_ylabel("count")

axes[1, 1].hist(true_counts_per, bins=range(0, max(true_counts_per + pred_counts_per + [1]) + 1), alpha=0.6, label="true")
axes[1, 1].hist(pred_counts_per, bins=range(0, max(true_counts_per + pred_counts_per + [1]) + 1), alpha=0.6, label="pred")
axes[1, 1].set_title(f"Labels per patient\ncoverage(any pred): {coverage:.2f} | hit(any correct): {hit_any:.2f}")
axes[1, 1].set_xlabel("# labels")
axes[1, 1].set_ylabel("count")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

if ml_metrics:
    print("Saved multi-label metrics:")
    for k, v in ml_metrics.items():
        print(f"  {k}: {v}")


### Cell 7 — Reference: numeric → Spanish infection-focus label map (`foco_map`)

Defines a lookup dictionary mapping the integer/float `foco` code stored in the dataset to its human-readable Spanish label (e.g. `1.0 → 'pulmonar'`, `4.0 → 'urinario'`). Import or reference this dict in downstream cells when you need to annotate plots or filter by infection focus.

In [ ]:
foco_map = {1.0: 'pulmonar',
 2.0: 'intraabdominal',
 3.0: 'biliar',
 4.0: 'urinario',
 5.0: 'cardiovascular',
 6.0: 'piel',
 7.0: 'sistema nervioso central',
 8.0: 'catéter venoso',
 9.0: 'vías altas respiratorias',
 10.0: 'osteoarticular',
 11.0: 'genital',
 12.0: 'desconocido'}